# SprintBoard Training with Hugging Face TRL (Google Colab)

This notebook runs GRPO training for SprintBoard using Hugging Face TRL via `train_grpo.py`.

Run cells top-to-bottom. Use a GPU runtime for best results.

## 1) Setup runtime and repo

If this notebook is outside the repository, keep `CLONE_REPO = True`.
If this notebook is already inside the repo in Colab, set it to `False`.

In [ ]:
import os
from pathlib import Path

CLONE_REPO = True
# Change this if you want to pull from a different remote.
REPO_URL = "https://huggingface.co/spaces/vikramsrini/sprint_planning_env"
WORKDIR = Path('/content/sprint_planning_env')

if CLONE_REPO:
    if WORKDIR.exists():
        !rm -rf /content/sprint_planning_env
    !git clone {REPO_URL} {WORKDIR}
else:
    WORKDIR = Path.cwd()

%cd {WORKDIR}
print('Working directory:', WORKDIR)


In [ ]:
# Optional: verify GPU
!nvidia-smi || true


## 2) Install dependencies

In [ ]:
# Core training dependencies
!pip -q install -U pip
!pip -q install -r requirements-train.txt

# Runtime deps required by SprintBoard environment used in reward functions
!pip -q install -r requirements.txt
# Make local package importable in Colab

!pip -q install -e .


## 3) (Optional) Hugging Face login
Only needed if you use private models or want to push checkpoints.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()


## 4) Configure training
Defaults below are tuned for Colab T4 (16GB) to reduce OOM risk. Increase gradually only after a successful run.

In [ ]:
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

MODEL_ID = 'unsloth/Qwen3-0.6B-unsloth-bnb-4bit'
OUTPUT_DIR = 'runs/colab-grpo-trl'

# Stability-first config. Lower LR + 1 epoch to avoid the epoch-2/3 reward
# collapse seen in earlier runs. Increase epochs ONLY if the reward is still
# climbing at the end of the run.
EPOCHS = 1
BATCH_SIZE = 4
GRAD_ACCUM = 2
NUM_GENERATIONS = 4
LEARNING_RATE = 3e-6
MAX_SAMPLES = 20
MAX_PROMPT_LENGTH = 512
MAX_COMPLETION_LENGTH = 256
LOGGING_STEPS = 1
# Set to a positive integer to cap training steps; leave as -1 to auto-infer
# from EPOCHS, MAX_SAMPLES, BATCH_SIZE, and GRAD_ACCUM.
MAX_STEPS = -1


## 5) Train with Hugging Face TRL (GRPO)
If you changed memory/env settings, restart runtime once before running this cell.

In [ ]:
!python train_grpo.py \
  --model-id "{MODEL_ID}" \
  --output-dir "{OUTPUT_DIR}" \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --grad-accum {GRAD_ACCUM} \
  --num-generations {NUM_GENERATIONS} \
  --learning-rate {LEARNING_RATE} \
  --max-samples {MAX_SAMPLES} \
  --max-prompt-length {MAX_PROMPT_LENGTH} \
  --max-completion-length {MAX_COMPLETION_LENGTH} \
  --logging-steps {LOGGING_STEPS} \
  --max-steps {MAX_STEPS}


## 6) Pick best checkpoint
Training saves one checkpoint per epoch. We pick the checkpoint with the highest
mean `reward_score_function` from the trainer log so we evaluate the best policy
rather than the last (which can drift downward).

In [ ]:
import json
from pathlib import Path

run_dir = Path(OUTPUT_DIR)
checkpoints = sorted(run_dir.glob('checkpoint-*'))
print('Found checkpoints:', [c.name for c in checkpoints])

best_ckpt = None
best_reward = float('-inf')
for ckpt in checkpoints:
    state_path = ckpt / 'trainer_state.json'
    if not state_path.exists():
        continue
    state = json.loads(state_path.read_text())
    rewards = [
        entry.get('rewards/reward_score_function/mean')
        for entry in state.get('log_history', [])
        if entry.get('rewards/reward_score_function/mean') is not None
    ]
    if not rewards:
        continue
    avg = sum(rewards[-10:]) / len(rewards[-10:])
    print(f'  {ckpt.name}: trailing-10 mean reward = {avg:.4f}')
    if avg > best_reward:
        best_reward = avg
        best_ckpt = ckpt

CHECKPOINT_DIR = str(best_ckpt) if best_ckpt else f'{OUTPUT_DIR}/checkpoint-final'
print('Selected checkpoint:', CHECKPOINT_DIR, 'with reward', best_reward)


## 7) Evaluate checkpoint (optional)

In [ ]:
# CHECKPOINT_DIR was selected in the previous cell (best by reward).
!python evaluate_training.py --output-dir runs/colab-eval --checkpoint-dir "{CHECKPOINT_DIR}"


## 8) Inspect artifacts

In [ ]:
from pathlib import Path

summary_path = Path(OUTPUT_DIR) / 'artifacts' / 'summary.json'
if summary_path.exists():
    print(summary_path.read_text())
else:
    print('No summary.json found yet.')

!ls -lah {OUTPUT_DIR}/artifacts || true
!ls -lah runs/colab-eval || true


## 9) (Optional) Save outputs to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/sprintboard-runs
# !cp -r {OUTPUT_DIR} /content/drive/MyDrive/sprintboard-runs/
# !cp -r runs/colab-eval /content/drive/MyDrive/sprintboard-runs/
